# Result Screen Investigation

Before I can do any sorts of scripts writing, I need them datas.

In [6]:
import numpy as np
import pandas as pd 
import pytesseract
import cv2 
import matplotlib.pylab as plt

screenshot_path = '../img/verse-results-1.png'
camshot_path = '../img/para-lost-cam.jpg'

In [7]:
ss = cv2.imread(screenshot_path, cv2.IMREAD_COLOR)
cv2.imshow("image", ss)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [8]:
ph = cv2.imread(camshot_path, cv2.IMREAD_COLOR)
cv2.imshow("image", ph)
cv2.waitKey(0)
cv2.destroyAllWindows()

### Moire Pattern:
Focused on Moire Pattern a bit simply because It is guaranteed to have a moire pattern, at least on most of the pictures that I took.

[Wikipedia](https://en.wikipedia.org/wiki/Moir%C3%A9_pattern)\
[Moiré Zero: An Efficient and High-Performance Neural Architecture for Moiré Removal](https://arxiv.org/html/2507.22407v1)

In [26]:
def detect_moire(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    f = np.fft.fft2(gray)
    fshift = np.fft.fftshift(f)
    magnitude = np.log(np.abs(fshift) + 1)
    
    h, w = magnitude.shape
    center_mask = np.zeros_like(magnitude)
    cv2.circle(center_mask, (w//2, h//2), min(h,w)//8, 1, -1)
    
    peripheral_energy = magnitude[center_mask == 0].mean()
    center_energy = magnitude[center_mask == 1].mean()
    
    return peripheral_energy / center_energy 

Well.... it doesn't seems to be the case here at least

In [27]:
ph_sample_path_2 = '../img/ph-sample-2.png'
ph_sample_path_3 = '../img/ph-sample-3.png'
ss_sample_path = '../img/results.png'

In [28]:
ph2 = cv2.imread(ph_sample_path_2, cv2.IMREAD_COLOR)
ph3 = cv2.imread(ph_sample_path_3, cv2.IMREAD_COLOR)
ss2 = cv2.imread(ss_sample_path, cv2.IMREAD_COLOR)

print(detect_moire(ph3))
print(detect_moire(ph2))
print(detect_moire(ph))

print(detect_moire(ss2))
print(detect_moire(ss))

0.7881848313621754
0.7867680634577151
0.734978969983676
0.7933173460363591
0.7785496013284077


Most probably since it is not too persistent.

### Perspective Distortion

I mean its pretty obvious.

In [29]:
def detect_perspective_distortion(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, 
                             threshold=100, minLineLength=100, maxLineGap=10)
    
    if lines is None:
        return 0
    
    angles = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        angle = np.degrees(np.arctan2(y2 - y1, x2 - x1))
        angles.append(angle % 180)
    
    # Screenshots have perfectly horizontal/vertical lines (0°, 90°)
    # Phone photos have slight tilts
    near_axis = sum(1 for a in angles if a % 90 < 3 or a % 90 > 87)
    distortion_score = 1 - (near_axis / len(angles))
    
    return distortion_score  # Higher = more likely phone photo

In [30]:
print(detect_perspective_distortion(ph3))
print(detect_perspective_distortion(ph2))
print(detect_perspective_distortion(ph))

print(detect_perspective_distortion(ss2))
print(detect_perspective_distortion(ss))

0.4186991869918699
0.3119777158774373
0.416819012797075
0.19098143236074272
0.14393939393939392


Seems like a better way to detect whether its photo or screenshot is by detecting the perspective distortion.

### Noise & Blur Profile

I have some doubt on this to be honest, mostly due to the fact that photograph doesn't always have more noise and blur.

In [31]:
def analyze_noise_profile(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Laplacian variance — screenshots are sharper
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    
    # Local standard deviation — phone photos have uneven noise
    kernel = np.ones((5,5), np.float32) / 25
    local_mean = cv2.filter2D(gray.astype(float), -1, kernel)
    local_sq_mean = cv2.filter2D((gray.astype(float))**2, -1, kernel)
    local_std = np.sqrt(np.abs(local_sq_mean - local_mean**2))
    noise_variance = local_std.std()
    
    return laplacian_var, noise_variance

In [32]:
print(analyze_noise_profile(ph3))
print(analyze_noise_profile(ph2))
print(analyze_noise_profile(ph))

print(analyze_noise_profile(ss2))
print(analyze_noise_profile(ss))

(np.float64(1209.906417766259), np.float64(16.62772059577871))
(np.float64(1281.478437916911), np.float64(15.46686996216868))
(np.float64(291.26113886193946), np.float64(12.35042895683182))
(np.float64(2359.764941529816), np.float64(21.784202987074835))
(np.float64(2489.3674941706317), np.float64(22.604613637990465))


In [33]:
def classify_image(image):
    
    moire_ratio = detect_moire(image)
    distortion = detect_perspective_distortion(image)
    sharpness, noise = analyze_noise_profile(image)
    
    # Heuristic scoring
    score = 0
    if moire_ratio > 0.85:   score += 2   # strong moire = phone
    if distortion > 0.20:    score += 2   # tilted lines = phone
    if sharpness < 2000:      score += 1   # blurry = phone
    if noise > 25:           score += 1   # noisy = phone
    
    return "phone_photo" if score >= 3 else "screenshot"

In [34]:
print(classify_image(ph3))
print(classify_image(ph2))
print(classify_image(ph))

print(classify_image(ss2))
print(classify_image(ss))

phone_photo
phone_photo
phone_photo
screenshot
screenshot


It seems to be working, beside for the moire detector. I can train a model even (I don't need to obvly)\
Now how do I get to extract the text from the photograph lol. It's a lot more easier for screenshot since its gonna be on the same place.